# 02 - Ingest TMDB credits
POC 31: CineScope. For every title in `raw_titles`, fetches credits and
movie runtimes, keeps directors plus top-3 billed cast, and writes the
`raw_credits` Delta table.

- Movies: `/movie/{id}?append_to_response=credits` (runtime + credits, one call)
- TV: `/tv/{id}/aggregate_credits` (crew dept Directing, cast by episode count)

Idempotent: on re-run, titles already present in `raw_credits` are skipped.

This product uses the TMDB API but is not endorsed or certified by TMDB.


In [ ]:
TMDB_API_KEY = ""  # paste key here for the POC run, clear before commit
TOP_CAST = 3
MAX_WORKERS = 8  # stays inside TMDB's ~40 req/s ceiling with headroom

assert TMDB_API_KEY, "Set TMDB_API_KEY before running"


In [ ]:
import requests, time

BASE = "https://api.themoviedb.org/3"

def tmdb_get(path, **params):
    params["api_key"] = TMDB_API_KEY
    for attempt in range(5):
        r = requests.get(f"{BASE}{path}", params=params, timeout=30)
        if r.status_code == 429:
            time.sleep(float(r.headers.get("Retry-After", 1)) + 0.1)
            continue
        if r.status_code == 404:
            return None
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"TMDB request failed after retries: {path}")


In [ ]:
# Work list: titles minus those already fetched (checkpoint for re-runs)
titles = spark.table("raw_titles").select("tmdbKey", "tmdbId", "mediaType").collect()

done = set()
if spark.catalog.tableExists("raw_credits"):
    done = {r.titleKey for r in spark.table("raw_credits").select("titleKey").distinct().collect()}
work = [t for t in titles if t.tmdbKey not in done]
print(f"{len(titles):,} titles, {len(done):,} already fetched, {len(work):,} to go")


In [ ]:
def person_fields(p):
    return {
        "personTmdbId": p["id"],
        "personName": (p.get("name") or "")[:300],
        "knownForDepartment": (p.get("known_for_department") or None),
        "profilePath": p.get("profile_path"),
    }

def fetch_movie(t):
    payload = tmdb_get(f"/movie/{t.tmdbId}", append_to_response="credits")
    if payload is None:
        return [], None
    rows = []
    credits = payload.get("credits", {})
    for c in credits.get("crew", []):
        if c.get("job") == "Director":
            rows.append({**person_fields(c), "titleKey": t.tmdbKey,
                         "category": "director", "ordering": 0, "characterName": None})
    for c in sorted(credits.get("cast", []), key=lambda x: x.get("order", 999))[:TOP_CAST]:
        rows.append({**person_fields(c), "titleKey": t.tmdbKey,
                     "category": "cast", "ordering": int(c.get("order", 0)),
                     "characterName": (c.get("character") or None)})
    return rows, payload.get("runtime")

def fetch_tv(t):
    payload = tmdb_get(f"/tv/{t.tmdbId}/aggregate_credits")
    if payload is None:
        return [], None
    rows, seen = [], set()
    for c in payload.get("crew", []):
        if c.get("department") == "Directing" and c["id"] not in seen:
            seen.add(c["id"])
            rows.append({**person_fields(c), "titleKey": t.tmdbKey,
                         "category": "director", "ordering": 0, "characterName": None})
    cast = sorted(payload.get("cast", []), key=lambda x: -(x.get("total_episode_count") or 0))[:TOP_CAST]
    for i, c in enumerate(cast):
        roles = c.get("roles") or []
        character = max(roles, key=lambda r: r.get("episode_count") or 0)["character"] if roles else None
        rows.append({**person_fields(c), "titleKey": t.tmdbKey,
                     "category": "cast", "ordering": i, "characterName": character})
    return rows, None


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

credit_rows, runtime_rows, failures = [], [], []

def fetch_one(t):
    return (t, *(fetch_movie(t) if t.mediaType == "movie" else fetch_tv(t)))

with ThreadPoolExecutor(MAX_WORKERS) as pool:
    futures = [pool.submit(fetch_one, t) for t in work]
    for i, f in enumerate(as_completed(futures), 1):
        try:
            t, rows, runtime = f.result()
            credit_rows.extend(rows)
            if runtime:
                runtime_rows.append({"tmdbKey": t.tmdbKey, "runtimeMinutes": int(runtime)})
        except Exception as e:
            failures.append(str(e))
        if i % 1000 == 0:
            print(f"{i:,}/{len(work):,} fetched, {len(failures)} failures")

print(f"Done: {len(credit_rows):,} credit rows, {len(runtime_rows):,} runtimes, {len(failures)} failures")


In [ ]:
# Append credits (checkpoint-friendly) and merge runtimes onto raw_titles
if credit_rows:
    spark.createDataFrame(pd.DataFrame(credit_rows)) \
        .write.mode("append").saveAsTable("raw_credits")

if runtime_rows:
    # raw_titles is built from /discover payloads which carry no runtime;
    # add the column on first arrival of runtime data
    if "runtimeMinutes" not in [f.name for f in spark.table("raw_titles").schema.fields]:
        spark.sql("ALTER TABLE raw_titles ADD COLUMNS (runtimeMinutes INT)")
    spark.createDataFrame(pd.DataFrame(runtime_rows)).createOrReplaceTempView("v_runtimes")
    spark.sql("""
        MERGE INTO raw_titles t USING v_runtimes r
        ON t.tmdbKey = r.tmdbKey
        WHEN MATCHED THEN UPDATE SET t.runtimeMinutes = r.runtimeMinutes
    """)

print(spark.table("raw_credits").groupBy("category").count().collect())
